# Geometry + YOLOv8 / MediaPipe + one posture network

Your **12 metrics remain explicit learned inputs**. Image features supplement ambiguous geometry, and the camera display draws real joints/vectors and all measured metrics. The separately versioned schema-3 angles preserve anatomical left/right direction relative to the torso; these are not physical roll angles. Old models retain their old feature definitions.

Use the same reviewed BabyTraining.zip, including actual overhead doll recordings when available. The original model and [exact group test ZIP](https://github.com/kg905/SeniorDesignProject/tree/main/artifacts/team_test_runs/colab-31p5-4094df16) remain available. [Workflow, definitions and Pi capture instructions](https://github.com/kg905/SeniorDesignProject/blob/main/docs/FUSION_GEOMETRY_V2.md).

Choose **Runtime → Change runtime type → GPU** before setup for faster training. CPU also works. Test images do not choose epochs, normalization, calibration or thresholds. This inspected pilot cannot establish 95% performance on unseen infants; failures count as errors. Download results before resetting Colab.


## 1. Get the code and original grouping reference
A sparse checkout avoids downloading all 5,814 source images again. Existing checkouts are fast-forwarded without deleting local data.

In [ ]:
import os, subprocess, sys
from pathlib import Path
ROOT = Path('/content/SeniorDesignProject')
GIT_ENV = dict(os.environ, GIT_LFS_SKIP_SMUDGE='1')
if not ROOT.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--filter=blob:none', '--sparse',
                    'https://github.com/kg905/SeniorDesignProject.git', str(ROOT)], env=GIT_ENV, check=True)
    subprocess.run(['git', 'sparse-checkout', 'set', 'src', 'training', 'deploy/single-model',
                    'docs', 'models', 'artifacts/dataset_audit_v2'], cwd=ROOT, env=GIT_ENV, check=True)
else:
    subprocess.run(['git', 'pull', '--ff-only'], cwd=ROOT, env=GIT_ENV, check=True)
os.chdir(ROOT)
import importlib
sys.path.insert(0, str(ROOT / "src"))
importlib.invalidate_caches()
print('Source ready:', ROOT)


## 2. Install dependencies

This affects this temporary Colab runtime, not your Pi. Keep only one pip cv2 distribution. If Colab requests a restart, restart after installation, rerun section 1, then continue to section 3. Do not skip an unsuccessful install.

Use Colab's matching preinstalled Torch/TorchVision pair; select a GPU runtime for faster training. Ultralytics is installed with --no-deps so it does not add a second OpenCV distribution. If a restart is required, do it before uploading data.


In [ ]:
%pip uninstall -y opencv-python opencv-python-headless opencv-contrib-python opencv-contrib-python-headless
%pip install -r /content/SeniorDesignProject/training/requirements.txt
%pip install --no-deps -e /content/SeniorDesignProject
!apt-get update -qq
!apt-get install -y -qq git-lfs libgles2 libegl1 libgl1 libportaudio2

%pip install -r /content/SeniorDesignProject/training/requirements-fusion-colab.txt
%pip install --no-deps ultralytics==8.3.200

import torch, torchvision
print('Torch:', torch.__version__, 'TorchVision:', torchvision.__version__)
print('GPU available:', torch.cuda.is_available())


## 3. Upload your sorted ZIP to LOCAL Colab storage

Upload BabyTraining.zip, not the original unsorted Roboflow ZIP. Upload, extraction and training stay under **/content/cribwatch-runs**, not the Google Drive mount. This prevents a disconnected Drive from interrupting active file writes. **Local files disappear on a runtime reset: download your model/results in section 8 before closing/resetting Colab.**

If the ZIP is already in Drive, set ZIP_IN_DRIVE below; only that option mounts Drive and copies the source ZIP locally. Otherwise leave it empty and select the ZIP from your computer. Existing Drive reviews/previous runs are not overwritten.


In [ ]:
import os, sys, subprocess, json
from pathlib import Path
ROOT = Path('/content/SeniorDesignProject')
os.chdir(ROOT)
import importlib
sys.path.insert(0, str(ROOT / "src"))
importlib.invalidate_caches()
from google.colab import files
from cribwatch.colab_io import local_zip_run
ZIP_IN_DRIVE = ''  # optional: '/content/drive/MyDrive/BabyTraining.zip'
if ZIP_IN_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
RUN, ZIP_PATH = local_zip_run(source_zip=ZIP_IN_DRIVE or None)
print('Verified local ZIP:', ZIP_PATH)
print('LOCAL run:', RUN)
print('Download section 8 outputs before resetting/disconnecting this runtime.')


## 4. Import folders as labels and check counts

The importer accepts one enclosing BabyTraining folder (or direct class folders). It verifies images, preserves your labels, keeps unknowns out of posture training, and quarantines conflicting copies. Hash matches to the original audit preserve known duplicate/source groups; new images get conservative duplicate grouping. None of this invents baby IDs.


In [ ]:
from cribwatch.assets import safe_unzip
from cribwatch.folder_data import import_folders
from cribwatch.data import read_csv
EXTRACTED = RUN / 'sorted-images'
safe_unzip(ZIP_PATH, EXTRACTED)
AUDIT = RUN / 'folder-audit'
summary = import_folders(EXTRACTED, AUDIT, reference_index=ROOT / 'artifacts/dataset_audit_v2/images.csv')
print('Your folder counts:', summary['folder_counts'])
print('Eligible exact-unique posture images:', summary['eligible_unique_counts'])
print('Matched to original audit:', summary['matched_original_images'])
print('Conservative source groups:', summary['source_groups'])
print('Conflicting images excluded:', summary['conflicting_images'])
print('Ignored files:', summary['ignored_files'][:10])
if summary['conflicting_images']:
    print('Inspect', AUDIT / 'conflicts.json', 'and correct the computer folders for a future run.')
if any(summary['eligible_unique_counts'].get(label, 0) == 0 for label in ['back','side','prone']):
    raise ValueError('Need non-conflicting images in ALL THREE posture folders before training.')
MANIFEST = AUDIT / 'images.csv'
print('Labels saved:', MANIFEST)


## 5. Extract the 12 metrics with directed torso angles

This requires fresh extraction because old axial angles discard direction. Generic YOLO weights are not infant-fine-tuned. Confidence is not ground-truth landmark accuracy.


In [ ]:
from cribwatch.assets import download_models, YOLO_ASSETS
from cribwatch.data import extract_manifest, read_csv
from cribwatch.fusion_model import body_evidence
from cribwatch.optimize import diagnose_features

download_models(ROOT / 'models', names=['pose_landmarker_lite.task', 'face_landmarker.task', 'yolov8n-pose.pt'])
ADAPTER = dict(kind='yolo_mediapipe_keypoints_v1', yolo_name='yolov8n-pose.pt',
               yolo_sha256=YOLO_ASSETS['yolov8n-pose.pt'][1], confidence=.25, padding=.2)
# Optional: a real, verified four-keypoint checkpoint trained ONLY on training groups.
TORSO4_WEIGHTS = ''
import hashlib, json, shutil
if TORSO4_WEIGHTS:
    source = Path(TORSO4_WEIGHTS)
    if source.name == 'yolov8n-pose.pt':
        raise ValueError('Give the fine-tuned model a distinct filename')
    with source.open('rb') as f:
        digest = hashlib.file_digest(f, 'sha256').hexdigest()
    target = ROOT / 'models' / source.name
    if target.exists() and target.resolve() != source.resolve():
        with target.open('rb') as f:
            if hashlib.file_digest(f, 'sha256').hexdigest() != digest:
                raise ValueError('Different weights share the same filename; use a new name')
    if target.resolve() != source.resolve():
        shutil.copy2(source, target)
    ADAPTER.update(yolo_name=source.name, yolo_sha256=digest, keypoint_layout='torso4', strategy='yolo_primary')
ADAPTER_ID = hashlib.sha256(json.dumps(ADAPTER, sort_keys=True).encode()).hexdigest()[:8]
FEATURES = RUN / f'features-directed-{ADAPTER_ID}.csv'
if not FEATURES.exists():
    extract_manifest(MANIFEST, FEATURES, ROOT / 'models', adapter=ADAPTER, feature_schema=3)
feature_rows = read_csv(FEATURES)
eligible = [r for r in feature_rows if r['training_eligible'] == '1']
diagnostics = diagnose_features(feature_rows)
print('Usable pose coverage:', f"{diagnostics['pose_coverage']:.1%}")
print('Body-evidence coverage:', f"{sum(body_evidence(r) for r in eligible)/len(eligible):.1%}")
for label, values in diagnostics['classes'].items():
    print(label, values)
print('Features saved:', FEATURES)


## 6. Train one network and measure the result

Both image and metric inputs are explicit. Epoch selection/calibration use validation only. Image augmentation changes color/brightness, not the coordinate geometry. Ablations measure sensitivity within the same trained network, not independent model trials.


In [ ]:
from cribwatch.fusion_train import train_fusion
MODEL = RUN / f'fusion-model-{ADAPTER_ID}'
report = train_fusion(FEATURES, MODEL, epochs=10, warmup=2, batch_size=32, device='auto')
print('OVERALL MODEL SCORE:', f"{100*report['accuracy_all_images_unknown_counts_as_error']:.1f}/100")
print('Prediction coverage:', f"{report['coverage']:.1%}")
for label, values in report['per_class'].items():
    print(label, 'recall:', f"{values['recall']:.1%}")
print('Body-evidence ceiling on this test:', f"{report['body_evidence_coverage_ceiling']:.1%}")
for branch, values in report['ablations'].items():
    print(branch, 'test score when branch is zeroed:', f"{100*values['accuracy_all_images_unknown_counts_as_error']:.1f}/100")
print('95% target established:', report['research_95pct_point_target_met'])
print('Reports:', MODEL)


## 7. See example scans with a pose box and risk level

These are examples after evaluation, not a new accuracy test. Body position maps to lower/moderate/high posture risk; unknown stays unknown. The visible-landmark box is not a separate baby/airway detector. Confidence is not a medical-risk percentage.


In [ ]:
examples = [r['image_path'] for r in read_csv(MODEL / 'test_predictions.csv')][:6]
ANNOTATED = RUN / 'annotated-scans'
subprocess.run([sys.executable, '-m', 'cribwatch.showcase', 'image', '--model', str(MODEL),
                '--models', str(ROOT / 'models'), '--showcase', '--output', str(ANNOTATED), *examples], check=True)
from IPython.display import display
from PIL import Image
for image in sorted(ANNOTATED.glob('*.jpg')):
    display(Image.open(image))

# Show the overall test score separately from each sample's confidence.
import json
model_report = json.loads((MODEL / 'test.json').read_text())
overall_score = 100 * model_report['accuracy_all_images_unknown_counts_as_error']
print(f'OVERALL MODEL SCORE (test accuracy): {overall_score:.1f}/100')
for scan in json.loads((ANNOTATED / 'predictions.json').read_text()):
    confidence = scan['confidence']
    score = f'{confidence:.1%}' if confidence > 0 else 'unavailable'
    print(f"{scan['image']}: {scan['posture']} | Per-image confidence: {score} | {scan['reason']}")


## 8. Download this model and matching Pi runtime

This separate export includes one fused posture network, 12-metric metadata, YOLO/MediaPipe assets, detailed geometry display and existing TCP 5000 alarm support. Read FUSION.md and HYBRID.md inside the ZIP. Physical Pi performance still needs testing.


In [ ]:
ZIP_OUT = RUN / 'SIDS_Pi5_Fusion_Geometry.zip'
subprocess.run([sys.executable, str(ROOT / 'deploy/single-model/build_handoff.py'),
                '--model', str(MODEL), '--output', str(ZIP_OUT)], check=True)
files.download(str(ZIP_OUT))
print('Local download source:', ZIP_OUT)
print('Keep the downloaded ZIP on your computer; /content storage is temporary.')
BACKUP_TO_DRIVE = False
if BACKUP_TO_DRIVE:
    from google.colab import drive
    import shutil
    drive.mount('/content/drive', force_remount=True)
    backup = Path('/content/drive/MyDrive/CribWatchManual') / RUN.name
    backup.mkdir(parents=True, exist_ok=True)
    shutil.copy2(ZIP_OUT, backup / ZIP_OUT.name)
    print('Copied to Drive:', backup / ZIP_OUT.name)


## Next data collection

Use actual labeled overhead doll clips and matching camera frames, not terminal screenshots. Preserve real session IDs across adjacent frames. Add reviewed recordings, import again into a NEW run, then retrain. Schema-3 directed angles require the matching model; do not replace the older classifier in place. Follow the linked workflow guide.
